In [9]:
from datetime import datetime, timedelta
import requests
import time
import pandas as pd
import holidays
from category_encoders import TargetEncoder
import pickle
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import FunctionTransformer
import xgboost as xgb
import numpy as np
import openmeteo_requests
import requests_cache
from retry_requests import retry
import nltk
from nltk.corpus import stopwords

# Functies om pipeline op te stellen

### Functie om kijkcijferdata inclusief weerdata op te halen van start tot einddatum

In [10]:
def clean_kijkcijfer_data(df):
    # Verwijder de kolom 'ranking', deze is nutteloos
    df.drop('ranking', axis=1, inplace=True)

    # Verwijder rijen met null waarden, deze zijn fout geregistreerd in de DB
    df.dropna(inplace=True)
        
    # Verwijder rijen waarvan tijdformaat nie overeenkomt met xx:xx:xx
    time_pattern = r'^\d{2}:\d{2}:\d{2}$'

    df = df[
        df["startTime"].str.match(time_pattern, na=False) & 
        df["rLength"].str.match(time_pattern, na=False)
    ].copy()

    # rLength omzetten naar seconden -> duration_sec
    # dateDiff omzetten naar datetime
    df['duration_sec'] = pd.to_timedelta(df['rLength']).dt.total_seconds()
    df['date'] = pd.to_datetime(df["dateDiff"]).dt.date

    # rateInK omzetten naar int -> viewers
    df['viewers'] = df['rateInK'].apply(lambda x: int(''.join(str(x).split('.'))))

    #startTime: 24:30:00 omzetten naar 00:30:00 en dateDiff 1 dag verhogen
    def fix_next_day(rij):
        time_parts = rij['startTime'].split(':')
        if int(time_parts[0]) >= 24:
            time_parts[0] = str(int(time_parts[0]) - 24).zfill(2)
            rij['date'] += timedelta(days=1)
        rij['startTime'] = ':'.join(time_parts)
        return rij

    df = df.apply(fix_next_day, axis=1)

    # timestamp kolom toevoegen op basis van dateDiff en startTime
    df['timestamp'] = pd.to_datetime(df['date'].astype(str) + ' ' + df['startTime'].astype(str))

    # Live: boolean maken, of getal houden?? 28 > 7 > 1 > 0 heeft groter getal ook correlatie met kijkcijfer???
    # Beslissing: Live kolom laten als int

    # Hour year month day toevoegen
    df['hour'] = pd.to_datetime(df['startTime']).dt.hour
    df['year'] = pd.to_datetime(df['date']).dt.year
    df['month'] = pd.to_datetime(df['date']).dt.month
    df['day'] = pd.to_datetime(df['date']).dt.day

    # Onnodige kolommen verwijderen
    df.drop(['startTime', 'rLength', 'rateInK'], axis=1, inplace=True)

    df = df[['timestamp', 'date', 'year', 'month', 'day', 'hour', 'channel', 'description', 'duration_sec', 'live', 'viewers']]

    df.rename(columns={'description': 'program'}, inplace=True)

    return df

def fetch_kijkcijfers(start_date, end_date):
    data_list = []
    print(f"Start fetching kijkcijfers van {start_date.date()} tot {end_date.date()}")

    # Loop door elke dag
    current_date = start_date
    while current_date <= end_date:
        datum = f"{current_date.year}-{current_date.month}-{current_date.day}"
        url = f"https://api.cim.be/api/cim_tv_public_results_daily_views?dateDiff={datum}&reportType=north"
        
        try:
            response = requests.get(url)
            if response.status_code == 200:
                data = response.json()
                programma_lijst = data.get('hydra:member', [])
                
                for programma in programma_lijst:
                    try:
                        data_list.append({
                            'dateDiff': programma.get('dateDiff'),
                            'ranking': programma.get('ranking'),
                            'description': programma.get('description'),
                            'channel': programma.get('channel'),
                            'startTime': programma.get('startTime'),
                            'rLength': programma.get('rLength'),
                            'rateInK': programma.get('rateInK'),
                            'live': programma.get('live')
                        })
                        
                    except Exception as e:
                        print(f"Fout bij verwerken programma op {datum}: {e}")
            else:
                print(f"Geen data voor {datum} (HTTP {response.status_code})")
                
        except Exception as e:
            print(f"Fout bij ophalen {datum}: {e}")
        
        current_date += timedelta(days=1)
    
    print(f"Einde fetching kijkcijfers")
    # Maak een dataframe van de data
    df = pd.DataFrame(data_list)

    # Doe een basis cleaning van deze data om te kunnen mergen
    df = clean_kijkcijfer_data(df)

    return df

def fetch_weather_data(start_date, end_date):
    print(f"Start fetching weerdata van {start_date.date()} tot {end_date.date()}")
    # Setup Open-Meteo API client with cache and retry on error
    cache_session = requests_cache.CachedSession('.cache', expire_after=3600)
    retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
    openmeteo = openmeteo_requests.Client(session=retry_session)

    # Geografisch middelpunt van Vlaanderen
    latitude = 51.037861
    longitude = 4.240528

    # Controleer of de data in het verleden of de toekomst ligt
    today = datetime.now().date()
    if end_date.date() < today:
        # Historische data ophalen
        url = "https://archive-api.open-meteo.com/v1/archive"
    else:
        # Toekomstige data ophalen
        url = "https://api.open-meteo.com/v1/forecast"

    # Controleer of de einddatum binnen het toegestane bereik ligt
    max_end_date = datetime(2025, 4, 15).date()
    if end_date.date() > max_end_date:
        print(f"Waarschuwing: Einddatum {end_date.date()} is buiten het toegestane bereik. Bijwerken naar {max_end_date}.")
        end_date = datetime.combine(max_end_date, datetime.min.time())

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date.strftime('%Y-%m-%d'),
        "end_date": end_date.strftime('%Y-%m-%d'),
        "hourly": [
            "temperature_2m",
            "weathercode",
            "precipitation",
            "rain",
            "snowfall",
            "cloudcover",
            "windspeed_10m"
        ],
        "daily": [
            "weather_code",
            "wind_speed_10m_max",
            "sunrise",
            "sunset",
            "daylight_duration",
        ],
        "timezone": "Europe/Brussels",
        "temperature_unit": "celsius",
        "precipitation_unit": "mm",
        "windspeed_unit": "kmh"
    }

    # Data ophalen
    response = retry_session.get(url, params=params)
    if response.status_code == 200:
        data = response.json()
        
        # Hourly data verwerken
        hourly = data["hourly"]
        hourly_df = pd.DataFrame({
            "time": pd.to_datetime(hourly["time"]),
            "temperature_2m": hourly["temperature_2m"],
            "weather_code": hourly["weathercode"],
            "precipitation": hourly["precipitation"],
            "rain": hourly["rain"],
            "snowfall": hourly["snowfall"],
            "cloudcover": hourly["cloudcover"],
            "windspeed_10m": hourly["windspeed_10m"]
        })
        
        # Daily dataframe
        daily = data["daily"]
        daily_df = pd.DataFrame({
            "date": pd.to_datetime(daily["time"]),
            "weather_code_daily": daily["weather_code"],
            "wind_speed_10m_max": daily["wind_speed_10m_max"],
            "sunrise": daily["sunrise"],
            "sunset": daily["sunset"],
            "daylight_duration": daily["daylight_duration"],
        })
        
        hourly_df["date"] = hourly_df["time"].dt.date
        
        # Maak date kolom om op te mergen
        hourly_df["date"] = pd.to_datetime(hourly_df["date"])
        daily_df["date"] = pd.to_datetime(daily_df["date"])
        
        # Merge hourly en daily data
        combined_df = hourly_df.merge(daily_df, on="date", how="left")
        
        combined_df["hour"] = combined_df["time"].dt.hour
        combined_df["day_of_week"] = combined_df["time"].dt.dayofweek
        combined_df["month"] = combined_df["time"].dt.month
        combined_df["year"] = combined_df["time"].dt.year
        
        print(f"Einde fetching weerdata")

        return combined_df
    else:
        print(f"Fout: {response.status_code}")
        print(response.text)
        return pd.DataFrame()

def fetch_historic_data(start_date, end_date):
    kijkcijfers = fetch_kijkcijfers(start_date, end_date)
    weerdata = fetch_weather_data(start_date, end_date)

    kijkcijfers['date'] = pd.to_datetime(kijkcijfers['date'])

    weerdata['time'] = pd.to_datetime(weerdata['time'])

    weerdata['date'] = pd.to_datetime(weerdata['time'].dt.date)
    weerdata['hour'] = weerdata['time'].dt.hour

    merged_df = pd.merge(kijkcijfers, weerdata, on=['date', 'hour'], how='left')

    merged_df.drop(columns=['date', 'time', 'month_y', 'year_y', 'live'], inplace=True)

    merged_df.rename(columns={'month_x': 'month', 'year_x': 'year'}, inplace=True)

    merged_df = merged_df[['timestamp', 'year', 'month', 'day', 'day_of_week', 'hour', 
                        'channel', 'program', 'duration_sec', 'viewers', 
                        'temperature_2m', 'weather_code', 'precipitation', 'rain', 
                        'snowfall', 'cloudcover', 'windspeed_10m', 
                        'wind_speed_10m_max', 'sunrise', 'sunset', 'daylight_duration']]
    
    merged_df.dropna(inplace=True)

    return merged_df

### Functie voor feature engineering

In [11]:
def get_season(date):
    if date.month in [3, 4, 5]:
        return 'spring'
    elif date.month in [6, 7, 8]:
        return 'summer'
    elif date.month in [9, 10, 11]:
        return 'autumn'
    else:
        return 'winter'

# Om gegeven data te formatteren
def reformat_data(df):
    df = df.copy()
    # Maak timestamp
    df['timestamp'] = pd.to_datetime(df['Datum'].astype(str) + ' ' + df['Start'].astype(str))
    df.drop(['Datum', 'Start'], axis=1, inplace=True)

    # Maak year month day day_of_week hour
    df['year'] = pd.to_datetime(df['timestamp']).dt.year
    df['month'] = pd.to_datetime(df['timestamp']).dt.month
    df['day'] = pd.to_datetime(df['timestamp']).dt.day
    df['day_of_week'] = pd.to_datetime(df['timestamp']).dt.dayofweek
    df['hour'] = pd.to_datetime(df['timestamp']).dt.hour

    # Maak duration_sec
    df['duration_sec'] = pd.to_timedelta(df['Duur']).dt.total_seconds()

    return df

def timestamp_feature_engineering(df):
    df['timestamp'] = pd.to_datetime(df['timestamp'])

    df['season'] = df['timestamp'].apply(get_season)

    # weekday toevoegen
    df['weekday'] = df['timestamp'].dt.weekday

    # uur toevoegen
    df['hour'] = df['timestamp'].dt.hour

    # dag toevoegen
    df['day'] = df['timestamp'].dt.day

    # maand toevoegen
    df['month'] = df['timestamp'].dt.month

    # isWeekend toevoegen
    df['isWeekend'] = df['weekday'].apply(lambda x: 1 if x in [4, 5] else 0)

    # isPrimeTime toevoegen
    df['isPrimeTime'] = df['hour'].apply(lambda x: 1 if x >= 18 and x <= 21 else 0)

    # Sunrise en sunset hour toevoegen ipv datetime
    df['sunrise_hour'] = df['sunrise'].apply(lambda x: pd.to_datetime(x).hour)
    df['sunset_hour'] = df['sunset'].apply(lambda x: pd.to_datetime(x).hour)

    df.drop(['sunrise', 'sunset'], axis=1, inplace=True)

    return df

def merge_with_weather_data(data):
    min_date = data['timestamp'].min()
    max_date = data['timestamp'].max()

    weerdata = fetch_weather_data(min_date, max_date)

    weerdata['time'] = pd.to_datetime(weerdata['time'])
    weerdata['date'] = pd.to_datetime(weerdata['time'].dt.date)
    weerdata['hour'] = weerdata['time'].dt.hour

    data['date'] = pd.to_datetime(data['timestamp'].dt.date)
    data['hour'] = data['timestamp'].dt.hour

    merged_df = pd.merge(data, weerdata, on=['date', 'hour'], how='left')

    merged_df.drop(columns=['date', 'time', 'month_y', 'year_y', 'day_of_week_y', 'Duur'], inplace=True)

    merged_df.rename(columns={
        'month_x': 'month', 
        'year_x': 'year', 
        'day_of_week_x': 'day_of_week',
        'Programma': 'program',
        'Zender': 'channel',
    }, inplace=True)

    merged_df = merged_df[['timestamp', 'year', 'month', 'day', 'day_of_week', 'hour', 
                    'channel', 'program', 'duration_sec',
                    'temperature_2m', 'weather_code', 'precipitation', 'rain', 
                    'snowfall', 'cloudcover', 'windspeed_10m', 
                    'wind_speed_10m_max', 'sunrise', 'sunset', 'daylight_duration']]

    return merged_df

def add_lag(df, n):
    for i in range(1, n+1):
        df[f'viewers_lag{i}'] = df.sort_values('timestamp').groupby('program')['viewers'].shift(i).ffill()
    return df

# Voeg lag features toe aan de to predict data met weerdata
def create_lag_features(merged_data_to_predict):
    end_date = merged_data_to_predict['timestamp'].max() - timedelta(days=1)
    start_date = end_date - timedelta(weeks=3)

    historic_data = fetch_historic_data(start_date, end_date)

    data_to_lag_on = pd.concat([merged_data_to_predict, historic_data], ignore_index=True)

    # Voeg lag features toe
    data_with_lag = add_lag(data_to_lag_on, 2)

    # Vul ontbrekende lagwaarden met 0 als er geen voorgaande entries zijn
    lag_columns = [col for col in data_with_lag.columns if 'viewers_lag' in col]
    data_with_lag[lag_columns] = data_with_lag[lag_columns].fillna(0)

    to_predict_with_lag = data_with_lag[data_with_lag['viewers'].isnull()].drop(columns=['viewers'])

    return to_predict_with_lag

# # Add one-hot encoding
# def one_hot_encode(df):
#     with open('./models2/one_hot_encoder.pkl', 'rb') as f:
#         encoder = pickle.load(f)

#     df_cat = df[['weather_code', 'season', 'channel', 'day_of_week']]
#     df_encoded = encoder.transform(df_cat)
#     return df_encoded

def one_hot_encode(df):
    # One hot encoding voor 'live', 'channel', 'weather_code', 'season'
    with open('./models2/one_hot_encoder.pkl', 'rb') as f:
        one_hot_encoder = pickle.load(f)
    
    df_cat = df[['weather_code', 'season', 'channel', 'day_of_week']]

    df_1hot = one_hot_encoder.transform(df_cat)

    one_hot_output = pd.DataFrame(df_1hot.toarray(), 
                                  columns=one_hot_encoder.get_feature_names_out(), 
                                  index=df_cat.index)

    # Voeg de one-hot encoded data toe en drop de originele kolommen
    df = df.drop(columns=['weather_code', 'season', 'channel', 'day_of_week'])
    df = pd.concat([df, one_hot_output], axis=1)
    
    return df

# remove stopwords
def remove_program_stopwords(df):

    nltk.download('stopwords')

    stop_words_nl = set(stopwords.words('dutch'))
    stop_words_en = set(stopwords.words('english'))
    stop_words = stop_words_nl.union(stop_words_en)

    extra_stopwoorden = {'aflevering', 'herhaling', 'van', 'het', 'de', 'with', '?', ','}
    stop_words.update(extra_stopwoorden)

    def remove_stopwords(text):
        words = text.lower().split()
        words_filtered = [word for word in words if word not in stop_words]
        return ' '.join(words_filtered)

    df['program'] = df['program'].apply(remove_stopwords)
    return df

# Add tf-idf vectorisatie
def tfidf_vectorization(df):
    with open('./models2/tfidf_vectorizer.pkl', 'rb') as f:
        vectorizer = pickle.load(f)

    tfidf_matrix = vectorizer.transform(df['program'])
    tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), 
                            columns=vectorizer.get_feature_names_out(), 
                            index=df.index)

    # Zorg ervoor dat de gegenereerde kolommen overeenkomen met de vectorizer
    expected_columns = vectorizer.get_feature_names_out()
    for col in expected_columns:
        if col not in tfidf_df.columns:
            tfidf_df[col] = 0

    tfidf_df = tfidf_df[expected_columns]

    df = pd.concat([df.reset_index(drop=True), tfidf_df.reset_index(drop=True)], axis=1)

    return df

def target_encoding(df):
    with open('./models2/target_encoder.pkl', 'rb') as f:
        target_encoder = pickle.load(f)

    # Target encoding for 'program'
    df['program_target_enc'] = target_encoder.transform(df['program'])

    return df

def preprocess(data):
    data = reformat_data(data)
    data = merge_with_weather_data(data)
    data = create_lag_features(data)
    data = timestamp_feature_engineering(data)
    data = one_hot_encode(data)
    data = remove_program_stopwords(data)
    data = tfidf_vectorization(data)
    data = target_encoding(data)

    data = data.drop(columns=['timestamp', 'program'])
    
    return data

# Uitvoeren

In [14]:
to_predict = pd.DataFrame({
    "Programma": [
        "THUIS", "HET 7 UUR-JOURNAAL", "MAN BIJT HOND", "BLOKKEN", "NIEUWS 19U VTM",
        "DE DAG VAN VANDAAG", "FAMILIE", "HUIS GEMAAKT", "GELUKKIG GESCHEIDEN", "HET 1 UUR-JOURNAAL"
    ],
    "Zender": ["VRT 1", "VRT 1", "VRT 1", "VRT 1", "VTM", "VRT 1", "VTM", "VTM", "VRT 1", "VRT 1"],
    "Datum": ["2025-04-02"] * 10,
    "Start": [
        "20:17:52", "19:00:04", "19:46:05", "18:29:07", "18:59:48",
        "21:37:08", "20:05:28", "20:41:26", "20:44:02", "13:00:04"
    ],
    "Duur": [
        "0:24:30", "0:43:18", "0:23:30", "0:28:30", "0:53:21",
        "0:53:26", "0:25:44", "1:05:45", "0:50:35", "0:27:33"
    ]
})

def predict_viewers(data):
    # Create a pipeline
    preprocess_pipeline = Pipeline([
        ('preprocess', FunctionTransformer(preprocess)),
        ('scaler', StandardScaler()),
    ])

    # Preprocess the data
    preprocessed_data = preprocess_pipeline.fit_transform(data)

    # Load the pre-trained model
    with open('./models2/tuned_xgb_model_randomsearch.pkl', 'rb') as f:
        model = pickle.load(f)

    # Convert preprocessed_data back to a DataFrame
    preprocessed_data_df = pd.DataFrame(preprocessed_data)

    # Make predictions using the model
    predictions = model.predict(preprocessed_data)

    # Add predictions as a column in the original data
    data['Predictions'] = predictions

    return data

# Call the function
# to_predict = predict_viewers(to_predict)
# to_predict

In [27]:
to_predict = pd.read_csv('./data/raw/tv_kijkcijfers_raw_TEST.csv')

to_predict = to_predict.rename(columns={
    'dateDiff' : 'Datum',
    'startTime' : 'Start',
    'rLength' : 'Duur',
    'channel' : 'Zender',
    'description' : 'Programma',
    'rateInK' : 'viewers',
})

to_predict = to_predict[['Programma', 'Zender', 'Datum', 'Start', 'Duur', 'viewers']]

to_predict['Datum'] = pd.to_datetime(to_predict['Datum']).dt.date

to_predict

,Programma,Zender,Datum,Start,Duur,viewers
0,HET 7 UUR-JOURNAAL,VRT 1,2025-02-26,19:00:04,00:45:45,1.011.301
1,THUIS,VRT 1,2025-02-26,20:21:09,00:24:12,1.003.454
2,MAN BIJT HOND,VRT 1,2025-02-26,19:48:28,00:21:00,836.969
3,IK VRAAG HET AAN,VRT 1,2025-02-26,20:47:05,01:05:23,816.356
4,DE DAG VAN VANDAAG,VRT 1,2025-02-26,21:54:17,00:48:07,703.609
...,...,...,...,...,...,...
315,ZONDER STERREN,PLAY4,2025-03-13,21:17:28,00:50:16,194.234
316,TER ZAKE,VRT CANVAS,2025-03-13,20:54:58,00:34:11,182.216
317,MILO,VTM,2025-03-13,18:23:27,00:24:59,160.072
318,EXPEDITIE GOORIS,VTM2,2025-03-13,21:50:03,00:57:46,156.376


In [28]:
X = to_predict[['Programma', 'Zender', 'Datum', 'Start', 'Duur']]
y = to_predict['viewers']

y_pred = predict_viewers(X)
y_pred

Start fetching weerdata van 2025-02-26 tot 2025-03-13
Einde fetching weerdata
Start fetching kijkcijfers van 2025-02-19 tot 2025-03-12
Einde fetching kijkcijfers
Start fetching weerdata van 2025-02-19 tot 2025-03-12


C:\Users\dylan\AppData\Local\Temp\ipykernel_20412\3706021764.py:42: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['hour'] = pd.to_datetime(df['startTime']).dt.hour


Einde fetching weerdata


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\dylan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
C:\Users\dylan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


,Programma,Zender,Datum,Start,Duur,Predictions
0,HET 7 UUR-JOURNAAL,VRT 1,2025-02-26,19:00:04,00:45:45,237849.765625
1,THUIS,VRT 1,2025-02-26,20:21:09,00:24:12,195955.031250
2,MAN BIJT HOND,VRT 1,2025-02-26,19:48:28,00:21:00,189239.171875
3,IK VRAAG HET AAN,VRT 1,2025-02-26,20:47:05,01:05:23,194632.843750
4,DE DAG VAN VANDAAG,VRT 1,2025-02-26,21:54:17,00:48:07,202629.015625
...,...,...,...,...,...,...
315,ZONDER STERREN,PLAY4,2025-03-13,21:17:28,00:50:16,202538.468750
316,TER ZAKE,VRT CANVAS,2025-03-13,20:54:58,00:34:11,204411.875000
317,MILO,VTM,2025-03-13,18:23:27,00:24:59,217284.500000
318,EXPEDITIE GOORIS,VTM2,2025-03-13,21:50:03,00:57:46,204155.296875


In [30]:
y

0      1.011.301
1      1.003.454
2        836.969
3        816.356
4        703.609
         ...    
315      194.234
316      182.216
317      160.072
318      156.376
319      141.980
Name: viewers, Length: 320, dtype: object

In [33]:
y_pred['Actual'] = y.str.replace('.', '', regex=False).astype(float)
y_pred['Difference'] = y_pred['Predictions'] - y_pred['Actual']

y_pred

,Programma,Zender,Datum,Start,Duur,Predictions,Actual,Difference
0,HET 7 UUR-JOURNAAL,VRT 1,2025-02-26,19:00:04,00:45:45,237849.765625,1011301.0,-773451.234375
1,THUIS,VRT 1,2025-02-26,20:21:09,00:24:12,195955.031250,1003454.0,-807498.968750
2,MAN BIJT HOND,VRT 1,2025-02-26,19:48:28,00:21:00,189239.171875,836969.0,-647729.828125
3,IK VRAAG HET AAN,VRT 1,2025-02-26,20:47:05,01:05:23,194632.843750,816356.0,-621723.156250
4,DE DAG VAN VANDAAG,VRT 1,2025-02-26,21:54:17,00:48:07,202629.015625,703609.0,-500979.984375
...,...,...,...,...,...,...,...,...
315,ZONDER STERREN,PLAY4,2025-03-13,21:17:28,00:50:16,202538.468750,194234.0,8304.468750
316,TER ZAKE,VRT CANVAS,2025-03-13,20:54:58,00:34:11,204411.875000,182216.0,22195.875000
317,MILO,VTM,2025-03-13,18:23:27,00:24:59,217284.500000,160072.0,57212.500000
318,EXPEDITIE GOORIS,VTM2,2025-03-13,21:50:03,00:57:46,204155.296875,156376.0,47779.296875


In [34]:
y_pred['Difference'].mean()

np.float64(-210300.31787109375)